# 第14回　総括
## 不確実な世界で、孤高の意思決定をするための道具として

統計学Ⅰ（B）　／　北星学園大学　最終回

14回かけて、君は「直感を補正する道具」を一つずつ手に入れてきた。今日はそれを総ざらいし、**これからデータに出会ったときの構え**を持ち帰る。

### 第1回に戻る ―― 今のあなたなら、どう考える？

第1回の最初、こう問われた：

> 「3つのドア。1つ選び、司会者がハズレを1つ開ける。**変えるべきか？**」
>
> あのときの直感：「2択だから50:50、変えても同じ」。

今のあなたは、**なぜ「変えるべき（約2/3）」なのか**、自分の言葉で説明できるはずだ。もう一度シミュレーションで確かめ、第1回からの成長を確認しよう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

# モンティ・ホール（第1回の回収）
rng = np.random.default_rng(0)
N = 10000
当たり = rng.integers(0, 3, N); 最初の選択 = rng.integers(0, 3, N)
変えない勝率 = (最初の選択 == 当たり).mean()
変える勝率   = (最初の選択 != 当たり).mean()
print(f"変えない: {変えない勝率:.1%}　変える: {変える勝率:.1%}")
print("最初に外す確率が2/3。司会者がハズレを開けても、その2/3は『変えれば当たり』に化ける。")

数値（変える＝約67%）は第1回と同じ。違うのは、**君がもう「なぜ」を説明できる**こと。これが14回の成果だ。

---
## 1. 14回の地図 ―― 何を手に入れたか

```
記述統計（手元のデータを要約する）
 ├─ 第2回 代表値（平均・中央値・最頻値）と、その嘘
 ├─ 第3回 ばらつき（分散・標準偏差）と正規分布という仮定
 └─ 第4回 Python（Pandas）で自分で計算する

関係を見る
 └─ 第5回 相関と因果（相関≠因果・交絡を疑う）

推測統計（一部から全体を推し量る）
 ├─ 第6回 標本と中心極限定理（なぜ一部で全体が分かる）
 ├─ 第7回 推定・信頼区間（95%の本当の意味）
 ├─ 第8回 仮説検定とt検定（p値の正しい読み方）
 └─ 第9回 分散分析（検定を繰り返す罠）

モデルで説明する
 ├─ 第10回 回帰分析（数式で予測・その限界）
 └─ 第11回 AIC（当てはまり vs 複雑さ・過学習）

もう一つの世界観・本質の抽出
 ├─ 第12回 ベイズ統計（データで信念を更新する）
 └─ 第13回 主成分分析（次元を削減して本質を見る）
```

バラバラの手法に見えるが、貫いているのは一つ ―― **「直感やまぐれに惑わされず、データから慎重に結論を引き出す」** という態度だ。

---
## 2. 統計的誤用チェックリスト（卒業後に使う武器）

これからデータやニュースに出会ったとき、この10問を自分に問いかけてほしい。各回で学んだことの結晶だ。

| # | 問い | 回 |
|---|---|---|
| 1 | **平均だけ**見ていないか？（中央値・分布の形は？） | 2・3 |
| 2 | **ばらつき**を無視していないか？（SDは？正規と仮定してよい？） | 3 |
| 3 | **相関を因果**と取り違えていないか？（第三の変数＝交絡は？） | 5 |
| 4 | **標本の取り方**は偏っていないか？（無作為か。数の多さは偏りを直さない） | 6 |
| 5 | **信頼区間**を「真値が95%で入る確率」と誤解していないか？ | 7 |
| 6 | **p値**を効果の大きさ／正しさの確率と混同していないか？ | 8 |
| 7 | **検定を繰り返して**偽陽性を増やしていないか？ | 9 |
| 8 | **R²の高さ**だけでモデルを過信していないか？ | 10 |
| 9 | **複雑なモデルを過学習**させていないか？（AICで複雑さを罰する） | 11 |
| 10 | **事前情報**を無視、または「主観だから」と切り捨てていないか？ | 12 |

（＋ PCA：次元削減では情報が一部失われ、軸の意味づけは人間の解釈、も忘れずに ― 第13回）

---
## 3. 卒業ミニ分析 ―― 道具を自分で使う

最後に、君自身がデータに問いを立てて答えてみよう。下は見本（**体重と行動圏**）。**列名を変えて、自分の問いに作り替えてよい。** そして、上のチェックリストで自分の結論を点検する。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    # 原典（PanTHERIA）から組み立て直す。まずリポジトリ同梱の複製、だめなら発行元から。
    # 発行元は User-Agent を見て弾くことがあるため、明示して取得する。
    import io, urllib.request
    SRCS = [
        "https://raw.githubusercontent.com/aonoa68/toukei-1/main/docs/data/PanTHERIA_1-0_WR05_Aug2008.txt.gz",
        "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt",
    ]
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = None
    for _url in SRCS:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=60) as _r:
                _raw = _r.read()
            _comp = "gzip" if _url.endswith(".gz") else None
            src = pd.read_csv(io.BytesIO(_raw), sep="\t", compression=_comp)
            break
        except Exception:
            continue
    if src is None:
        raise RuntimeError("原典データを取得できませんでした")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

# ── 見本：体重と行動圏に関係はあるか ──────────────────────
e = df.dropna(subset=["体重g", "行動圏km2"])
x, y = np.log10(e["体重g"]), np.log10(e["行動圏km2"])
r = x.corr(y)

print(f"体重 と 行動圏 の相関係数 r = {r:.2f}   (n={len(e)}種)")
print(f"体重の平均 {e['体重g'].mean():,.0f}g / 中央値 {e['体重g'].median():,.0f}g  ← #1 分布の形を見る")

plt.figure(figsize=(6.5, 4.5))
plt.scatter(x, y, s=16, alpha=0.5, color="#00897b")
plt.xticks([2,3,4,5], ["100g","1kg","10kg","100kg"])
plt.yticks([-2,-1,0,1], ["0.01km²","0.1km²","1km²","10km²"])
plt.xlabel("体重（対数）"); plt.ylabel("行動圏（対数）")
plt.title(f"体重と行動圏（r = {r:.2f}）")
plt.show()

> 結論を書く前に、チェックリストを通そう。例：
>
> 「r=0.68で相関はある。でも**平均だけ見ていないか**（#1）――この153種の体重は平均6,158g・中央値3,978gで大きく歪んでいたので、**対数で扱った**。**相関≠因果**（#3）――体が大きいから広く動くのか、広く動く生活だから体が大きくなったのか、この図では決められない。そもそも**第5回で見たとおり、体サイズは何にでも効く交絡**である。**標本の偏り**（#4）――使ったのは『行動圏が測られた153種』だけ。全376種の4割にすぎず、**体重だけなら記録がある265種の平均5,881gとも違う**。よく研究された種に偏っている。だから『体が大きいほど広く動く』という**傾向はあるが、原因とは言い切らないし、全霊長類に一般化もしない**」
>
> ――こう書ければ、君はもう統計の使い手だ。

---
## 最後に

統計は、**確実な答えをくれる魔法ではない**。p値も信頼区間もモデルも、不確実性を消してはくれない。できるのは、不確実性の大きさを**測り**、どこまで言えてどこから言えないかの**線を引く**ことだけだ。

だが、それで十分だ。なぜなら世界はもともと不確実で、それでも我々は判断しなければならないから。

直感は速いが、系統的に間違える（第1回）。空気や同調は、独立性を壊す。**統計は、群れず・直感に流されず・データに基づいて、自分で判断するための道具**だ。

> 不確実な世界で、孤高の意思決定をするための道具として ―― これを、君の手に渡す。
>
> 14回、おつかれさま。

**課題（Moodle）**：第1回の自分と今の自分を比べた総合ふりかえり＋（任意）自由テーマのミニ分析。